In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

unipd_deep_learning_2026_challenge_1_path = kagglehub.competition_download('unipd-deep-learning-2026-challenge-1')

print('Data source import complete.')


In [ ]:
import os
import time
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

In [ ]:
class CelebDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.data_frame = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, idx):
        img_name = os.path.join(self.img_dir, self.data_frame.iloc[idx, 0])
        image = Image.open(img_name).convert('RGB')
        labels = self.data_frame.iloc[idx, 1:].values.astype('float32')
        if self.transform:
            image = self.transform(image)
        return image, labels

transform = transforms.Compose([
    transforms.ToTensor(),
])

CHALLENGE_DIR = '/kaggle/input/competitions/unipd-deep-learning-2026-challenge-1'
TRAIN_CSV_PATH = f'{CHALLENGE_DIR}/train_data.csv'
TRAIN_IMG_DIR  = f'{CHALLENGE_DIR}/train_images'

train_dataset = CelebDataset(
    csv_file=TRAIN_CSV_PATH,
    img_dir=TRAIN_IMG_DIR,
    transform=transform
)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
print(f"DataLoader successfully initialized with {len(train_dataset)} training samples.")


DataLoader successfully initialized with 20000 training samples.


In [ ]:
# CARICAMENTO DATASET
full_df = pd.read_csv(TRAIN_CSV_PATH)
# RINOMINIAMO LA COLONNA PERCHè CI SERVE IN FORMATO 1234.jpg
full_df['id'] = full_df['id'].apply(lambda x: f"{str(x).zfill(5)}.jpg")

# DIVIDIAMO IN TRAINING SET E VALIDATION SET
train_df, val_df = train_test_split(full_df, test_size=0.2, stratify=full_df['celebrity_id'], random_state=42)

train_df.to_csv('train_split.csv', index=False)

# AGGIUNGO PESI PERCHè IL DATASET è SBILANCIATO
class_counts = train_df['celebrity_id'].value_counts()
weights = 1.0 / class_counts
sample_weights = train_df['celebrity_id'].map(weights).values

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True)


val_df.to_csv('val_split.csv', index=False)

# APPLICHIAMO TRASFORMAZIONI AL DATASET USATO PER IL TRAINING PER RENDERE IL MODELLO PIù ROBUSTO

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]
IMG_SIZE = 128

transform_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.15))
])
# NORMALIZZIAMO E RENDIAMO UN TENSORE ANCHE IL VALIDATION SET
transform_val = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

BATCH_SIZE = 64

train_loader = DataLoader(
    CelebDataset('train_split.csv', TRAIN_IMG_DIR, transform_train),
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=2,
    pin_memory=True
)
val_loader = DataLoader(
    CelebDataset('val_split.csv', TRAIN_IMG_DIR, transform_val),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
class ResBlock(nn.Module):
    """Blocco residuo standard con shortcut connection."""
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, stride, 1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_c)
        )
        self.shortcut = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_c, out_c, 1, stride, bias=False),
                nn.BatchNorm2d(out_c)
            )

    def forward(self, x):
        return F.relu(self.conv(x) + self.shortcut(x), inplace=True)


class AT_Net(nn.Module):
    """
    Rete multi-task per attributi (8 classi binarie) + identità (501 classi).
    Backbone ResNet custom con 5 blocchi residui → 1024 feature.
    Teste MLP con Dropout per regolarizzazione.
    """
    def __init__(self, num_attrs=8, num_ids=501, dropout=0.4):
        super().__init__()
        self.backbone = nn.Sequential(
            # Strato di ingresso: 3 canali RGB → 64 mappe
            nn.Conv2d(3, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            # 5 blocchi residui — ogni stride=2 dimezza la risoluzione
            ResBlock(64,  128, stride=2),   # 128x128 → 64x64
            ResBlock(128, 256, stride=2),   # 64x64  → 32x32
            ResBlock(256, 512, stride=2),   # 32x32  → 16x16
            ResBlock(512, 512, stride=2),   # 16x16  → 8x8
            ResBlock(512, 1024, stride=2),  # 8x8    → 4x4  ← blocco aggiunto
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()                    # → vettore 1024
        )
        # Testa attributi: MLP a 2 livelli
        self.attr_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(1024, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_attrs)
        )
        # Testa identità: MLP a 2 livelli (più capiente)
        self.id_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, num_ids)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.attr_head(features), self.id_head(features)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AT_Net().to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parametri trainabili: {total_params:,}')

Parametri trainabili: 25,540,413


In [ ]:
# --- pos_weight per gli attributi sbilanciati ---
# Es. 'Wearing_Hat' è raro → pesiamo le classi positive
ATTR_COLS = ['No_Beard', 'Young', 'Mouth_Slightly_Open', 'Smiling',
             'Male', 'Wavy_Hair', 'Black_Hair', 'Wearing_Hat']

pos_counts = train_df[ATTR_COLS].sum()
neg_counts = len(train_df) - pos_counts
pos_weight = torch.tensor((neg_counts / pos_counts).values, dtype=torch.float32).to(device)
print('pos_weight attributi:')
for col, w in zip(ATTR_COLS, pos_weight.cpu()):
    print(f'  {col:25s}: {w:.2f}')

# BCEWithLogitsLoss con peso per attributi sbilanciati
attr_loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# CrossEntropyLoss con label smoothing per l'identità (501 classi)
id_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

# --- Ottimizzatore AdamW con weight decay ---
NUM_EPOCHS = 40
LR = 3e-4

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

# Cosine Annealing: abbassa gradualmente il LR verso eta_min
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
    eta_min=1e-5
)

print(f'\nEpoche: {NUM_EPOCHS} | LR iniziale: {LR} | Scheduler: CosineAnnealing')

pos_weight attributi:
  No_Beard                 : 0.18
  Young                    : 0.24
  Mouth_Slightly_Open      : 1.16
  Smiling                  : 1.18
  Male                     : 1.78
  Wavy_Hair                : 1.86
  Black_Hair               : 3.06
  Wearing_Hat              : 24.81

Epoche: 40 | LR iniziale: 0.0003 | Scheduler: CosineAnnealing


In [ ]:
def evaluate_model(model, loader, device):
    model.eval()
    attr_preds, attr_true = [], []
    id_preds,   id_true   = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs   = imgs.to(device)
            labels = labels.to(device)

            out_a, out_i = model(imgs)

            # Attributi: sigmoid + soglia 0.5
            attr_preds.append((torch.sigmoid(out_a) > 0.5).int().cpu().numpy())
            attr_true.append(labels[:, :-1].int().cpu().numpy())

            # Identità: argmax
            id_preds.append(torch.argmax(out_i, dim=1).cpu().numpy())
            id_true.append(labels[:, -1].long().cpu().numpy())

    attr_preds = np.vstack(attr_preds)
    attr_true  = np.vstack(attr_true)
    id_preds   = np.concatenate(id_preds)
    id_true    = np.concatenate(id_true)

    f1_attr = f1_score(attr_true, attr_preds, average='macro', zero_division=0)
    f1_id   = f1_score(id_true,   id_preds,   average='macro', zero_division=0)
    score   = 0.5 * f1_attr + 0.5 * f1_id

    return score, f1_attr, f1_id

In [ ]:
best_score    = 0.0
best_path     = 'best_celeb_model.pth'
LOSS_W_ATTR = 0.4
LOSS_W_ID   = 0.6

print(f'Inizio training per {NUM_EPOCHS} epoche...\n')

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    t0 = time.time()
    running_loss = 0.0

    for imgs, labels in train_loader:
        imgs   = imgs.to(device)
        labels = labels.to(device)

        attr_labels = labels[:, :-1]
        id_labels   = labels[:, -1].long()

        optimizer.zero_grad()
        out_a, out_i = model(imgs)

        loss = LOSS_W_ATTR * attr_loss_fn(out_a, attr_labels) + \
               LOSS_W_ID   * id_loss_fn(out_i,   id_labels)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()
    elapsed = time.time() - t0

    score, f1_attr, f1_id = evaluate_model(model, val_loader, device)
    avg_loss = running_loss / len(train_loader)
    current_lr = scheduler.get_last_lr()[0]

    print(f'Epoca {epoch:3d}/{NUM_EPOCHS} | '
          f'Score: {score:.4f} (Attr: {f1_attr:.3f}, ID: {f1_id:.3f}) | '
          f'LR: {current_lr:.2e} | '
          f'{elapsed:.1f}s')

    if score > best_score:
        best_score = score
        torch.save(model.state_dict(), best_path)
        print(f'  → Modello salvato (score: {best_score:.4f})')

print(f'\n Training completato. Best score: {best_score:.4f}')

Inizio training per 40 epoche...

Epoca   1/40 | Loss: 3.8917 | Score: 0.3208 (Attr: 0.640, ID: 0.002) | LR: 3.00e-04 | 81.0s
  → Modello salvato (score: 0.3208)
Epoca   2/40 | Loss: 3.5711 | Score: 0.3478 (Attr: 0.684, ID: 0.012) | LR: 2.98e-04 | 84.0s
  → Modello salvato (score: 0.3478)
Epoca   3/40 | Loss: 3.2378 | Score: 0.3929 (Attr: 0.754, ID: 0.032) | LR: 2.96e-04 | 84.1s
  → Modello salvato (score: 0.3929)
Epoca   4/40 | Loss: 2.9760 | Score: 0.4147 (Attr: 0.754, ID: 0.075) | LR: 2.93e-04 | 84.3s
  → Modello salvato (score: 0.4147)
Epoca   5/40 | Loss: 2.7305 | Score: 0.4240 (Attr: 0.748, ID: 0.100) | LR: 2.89e-04 | 84.2s
  → Modello salvato (score: 0.4240)
Epoca   6/40 | Loss: 2.5112 | Score: 0.4447 (Attr: 0.741, ID: 0.149) | LR: 2.84e-04 | 84.4s
  → Modello salvato (score: 0.4447)
Epoca   7/40 | Loss: 2.3115 | Score: 0.5096 (Attr: 0.817, ID: 0.202) | LR: 2.79e-04 | 84.2s
  → Modello salvato (score: 0.5096)
Epoca   8/40 | Loss: 2.1270 | Score: 0.5098 (Attr: 0.781, ID: 0.238) |

In [ ]:
for epoch in range(1, 11):
    model.train()
    t0 = time.time()
    running_loss = 0.0

    for imgs, labels in train_loader:
        imgs   = imgs.to(device)
        labels = labels.to(device)

        attr_labels = labels[:, :-1]
        id_labels   = labels[:, -1].long()

        optimizer.zero_grad()
        out_a, out_i = model(imgs)

        loss = LOSS_W_ATTR * attr_loss_fn(out_a, attr_labels) + \
               LOSS_W_ID   * id_loss_fn(out_i,   id_labels)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()
    elapsed = time.time() - t0

    score, f1_attr, f1_id = evaluate_model(model, val_loader, device)
    avg_loss = running_loss / len(train_loader)
    current_lr = scheduler.get_last_lr()[0]

    print(f'Epoca {epoch:3d}/10 | '
          f'Score: {score:.4f} (Attr: {f1_attr:.3f}, ID: {f1_id:.3f}) | '
          f'LR: {current_lr:.2e} | '
          f'{elapsed:.1f}s')

    if score > best_score:
        best_score = score
        torch.save(model.state_dict(), best_path)
        print(f'  → Modello salvato (score: {best_score:.4f})')

print(f'\n Training completato. Best score: {best_score:.4f}')

Epoca   1/40 | Score: 0.6973 (Attr: 0.868, ID: 0.527) | LR: 1.04e-05 | 84.2s
Epoca   2/40 | Score: 0.6965 (Attr: 0.868, ID: 0.525) | LR: 1.18e-05 | 84.3s
Epoca   3/40 | Score: 0.6979 (Attr: 0.868, ID: 0.528) | LR: 1.40e-05 | 84.4s
Epoca   4/40 | Score: 0.6980 (Attr: 0.867, ID: 0.529) | LR: 1.71e-05 | 84.4s
Epoca   5/40 | Score: 0.7017 (Attr: 0.870, ID: 0.533) | LR: 2.10e-05 | 84.3s
  → Modello salvato (score: 0.7017)
Epoca   6/40 | Score: 0.6976 (Attr: 0.868, ID: 0.527) | LR: 2.58e-05 | 84.3s
Epoca   7/40 | Score: 0.6996 (Attr: 0.866, ID: 0.533) | LR: 3.14e-05 | 84.3s
Epoca   8/40 | Score: 0.6988 (Attr: 0.867, ID: 0.530) | LR: 3.77e-05 | 84.5s
Epoca   9/40 | Score: 0.7002 (Attr: 0.864, ID: 0.537) | LR: 4.47e-05 | 84.4s
Epoca  10/40 | Score: 0.6993 (Attr: 0.869, ID: 0.530) | LR: 5.25e-05 | 84.4s

 Training completato. Best score: 0.7017


In [ ]:
# Carica il miglior modello
CHALLENGE_DIR = '/kaggle/input/competitions/unipd-deep-learning-2026-challenge-1'

TEST_IMG_DIR = f'{CHALLENGE_DIR}/test_images'

model_inf = AT_Net().to(device)
model_inf.load_state_dict(torch.load(best_path, map_location=device))
model_inf.eval()
print('Pesi del miglior modello caricati.')

# Prepara submission
train_df_sample = pd.read_csv(TRAIN_CSV_PATH, nrows=1)
columns = train_df_sample.columns

test_images = sorted(os.listdir(TEST_IMG_DIR))
submission_df = pd.DataFrame(columns=columns)
submission_df[columns[0]] = test_images
for col in columns[1:]:
    submission_df[col] = 0

preprocess = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

print(f'Inferenza su {len(test_images)} immagini...')
with torch.no_grad():
    for i in range(len(submission_df)):
        img_name  = submission_df.iloc[i, 0]
        full_path = os.path.join(TEST_IMG_DIR, img_name)

        if os.path.exists(full_path):
            img   = Image.open(full_path).convert('RGB')
            img_t = preprocess(img).unsqueeze(0).to(device)

            out_attr, out_id = model_inf(img_t)

            id_predetto   = torch.argmax(out_id, dim=1).item()
            attr_predetti = (torch.sigmoid(out_attr) > 0.5).int().cpu().numpy()[0]

            submission_df.iloc[i, -1]   = id_predetto
            submission_df.iloc[i, 1:-1] = attr_predetti
        else:
            print(f'  ATTENZIONE: {img_name} non trovato')

# Rimuove l'estensione dall'id
submission_df['id'] = submission_df['id'].str.split('.').str[0]

OUTPUT_FILE = '/kaggle/working/submission.csv'
submission_df.to_csv(OUTPUT_FILE, index=False)
print(f'\n Submission salvata: {OUTPUT_FILE}')
print(f'Righe: {len(submission_df)} | Colonne: {len(submission_df.columns)}')
submission_df.head()

Pesi del miglior modello caricati.
Inferenza su 4000 immagini...

 Submission salvata: /kaggle/working/submission.csv
Righe: 4000 | Colonne: 10


,id,No_Beard,Young,Mouth_Slightly_Open,Smiling,Male,Wavy_Hair,Black_Hair,Wearing_Hat,celebrity_id
0,20000,1,1,0,1,0,1,0,0,120
1,20001,0,1,1,1,1,0,1,0,220
2,20002,1,1,0,0,0,1,0,0,440
3,20003,1,0,1,1,1,0,0,0,273
4,20004,1,1,0,0,1,0,0,0,23
